# Training

Перша клітинка тягне закомічений код з гілки `BRANCH` і готує Colab (залежності, дані, W&B).
Далі кожна клітинка — окремий запуск: міняй гіперпараметри в `Config(...)` і запускай.

Ключі: скопіюй `.env.example` → `.env`, заповни і завантаж `.env` у `/content/` перед першою клітинкою.

In [ ]:
BRANCH = "feat/flair-bilstm"
REPO = "https://github.com/mykhailoisupov/RNN_for_NER.git"
ROOT = "/content/RNN_for_NER"

import os, subprocess, sys

if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)
for cmd in (["fetch", "-q", "origin"], ["checkout", "-q", BRANCH], ["reset", "-q", "--hard", f"origin/{BRANCH}"]):
    subprocess.run(["git", "-C", ROOT, *cmd], check=True)
print(subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

os.chdir(ROOT)
sys.path.insert(0, ROOT)
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]  # щоб після нового pull імпортувався свіжий код

from src.colab import setup
setup()

In [ ]:
from src.config import Config
from src.train import run

## Smoke test

Маленький шматок даних, без W&B — перевірити, що все запускається.

In [ ]:
run(Config(embedding="flair", rnn="lstm", train_size=2000, epochs=1, batch_size=32, wandb_mode="disabled", name="smoke"))

## Flair → BiLSTM

In [ ]:
cfg = Config(
    embedding="flair",
    rnn="lstm",
    hidden_size=256,
    num_layers=2,
    dropout=0.3,
    lr=1e-3,
    batch_size=32,
    epochs=5,
    name="flair-bilstm",
)
run(cfg)

## fastText → BiGRU

In [ ]:
cfg = Config(
    embedding="fasttext",
    rnn="gru",
    hidden_size=256,
    num_layers=2,
    dropout=0.3,
    lr=1e-3,
    batch_size=128,
    epochs=5,
    name="fasttext-bigru",
)
run(cfg)